# Notebook 4 — Preprocessing and Dataset Preparation

**Project:** P63 – Multimodal Deep Learning for Autoimmune Disease Diagnosis  
**Phase:** Rheumatoid Arthritis (RA) only  
**Dataset:** RAM-H1200-v1  

---

## What this notebook does

This is the **final notebook** in the preprocessing pipeline. It takes everything discovered in Notebooks 1–3 and produces a clean, ready-to-train dataset.

**What is done here — and why:**

| Step | What | Why |
|---|---|---|
| 1 | Load and filter manifest | Remove 46 rows with no image on disk |
| 2 | Exclude corrupted images | Prevent silent errors during training |
| 3 | Parse `PixelSpacing` | Convert string `[0.15, 0.15]` → float `0.15` |
| 4 | Parse `ImageSize` | Standardise mixed string formats |
| 5 | Encode categorical features | Sex, LR, Center → numeric |
| 6 | Normalise `Age` | Subtract training mean, divide by training std |
| 7 | Assemble final metadata feature table | Safe features only, no target leakage |
| 8 | Validate image preprocessing pipeline | Test resize + normalise on one image |
| 9 | Compute per-channel normalisation stats | Mean and std from training images only |
| 10 | Verify patient isolation across splits | Confirm no leakage |
| 11 | Save all outputs | CSV files consumed by the PyTorch Dataset class |

**Prerequisite:** Run Notebooks 01, 02, and 03 first.

**Outputs saved:**
- `outputs/reports/train_manifest.csv`  
- `outputs/reports/val_manifest.csv`  
- `outputs/reports/test_manifest.csv`  
- `outputs/reports/normalisation_stats.csv` — mean and std per channel (training set only)  
- `outputs/reports/label_encoders.csv` — encoding mappings for categorical columns  
- `outputs/reports/04_preprocessing_report.csv` — summary of all preprocessing steps

---
## 0. Imports and configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import json
import warnings
from pathlib import Path

# ── Data handling ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Image processing ──────────────────────────────────────────────────────────
from PIL import Image
import PIL

# ── Machine learning utilities ────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Suppress warnings ─────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
Image.MAX_IMAGE_PIXELS = None

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid', palette='muted')

print('All imports successful.')

In [ ]:
# ── Path configuration ────────────────────────────────────────────────────────
PROJECT_ROOT = Path(os.getcwd()).parent

MANIFEST_CSV     = PROJECT_ROOT / 'outputs' / 'reports' / 'image_manifest.csv'
IMG_STATS_CSV    = PROJECT_ROOT / 'outputs' / 'reports' / '02_image_stats.csv'
OUTPUTS_DIR      = PROJECT_ROOT / 'outputs' / 'reports'
PLOTS_DIR        = PROJECT_ROOT / 'outputs' / 'plots'

# ── Preprocessing hyper-parameters ───────────────────────────────────────────
# These are the settings used throughout; change here to affect the whole notebook.
TARGET_SIZE     = (224, 224)     # (width, height) to resize all images
RESIZE_METHOD   = Image.BILINEAR # PIL resampling filter (good quality/speed tradeoff)
PIXEL_SCALE     = 255.0          # divide raw 0-255 values by this → [0, 1]

# Ensure required inputs exist
for p in [MANIFEST_CSV]:
    assert p.exists(), f'Required file not found: {p}\nPlease run the earlier notebooks first.'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('TARGET_SIZE  :', TARGET_SIZE)
print('Paths OK.')

---
## 1. Load the image manifest and image stats

In [ ]:
# ── Load manifest (all 1200 images, with merged metadata) ─────────────────────
manifest = pd.read_csv(MANIFEST_CSV)
print(f'Manifest loaded: {len(manifest)} rows')
print(f'Columns: {list(manifest.columns)}')
print()

# ── Load image stats from Notebook 02 (if available) ─────────────────────────
# We use these to exclude any corrupted images.
if IMG_STATS_CSV.exists():
    img_stats = pd.read_csv(IMG_STATS_CSV)
    corrupted_files = set(
        img_stats[img_stats['status'] == 'corrupted']['filename'].tolist()
    )
    print(f'Image stats loaded. Corrupted images found: {len(corrupted_files)}')
else:
    corrupted_files = set()
    print('Image stats file not found — skipping corruption filter.')
    print('(Run Notebook 02 to enable this step.)')

---
## 2. Filter out corrupted images

In [ ]:
# ── Remove corrupted images (if any were detected in Notebook 02) ─────────────
before = len(manifest)
if corrupted_files:
    manifest = manifest[~manifest['filename'].isin(corrupted_files)].copy()
    print(f'Removed {before - len(manifest)} corrupted images.')
else:
    print('No corrupted images to remove.')

print(f'Manifest after corruption filter: {len(manifest)} rows')

---
## 3. Parse `PixelSpacing` — string to float

The raw value looks like `[0.15, 0.15]`. We extract the numeric value.
Both values are always equal (isotropic spacing), so we keep just one.

In [ ]:
def parse_pixel_spacing(raw_value) -> float:
    """
    Parse PixelSpacing from its raw string representation to a float.

    Handles formats like:
        '[0.15, 0.15]'   → 0.15
        '0.15'           → 0.15
        0.15  (already float) → 0.15
    Returns NaN if parsing fails.
    """
    if pd.isnull(raw_value):
        return np.nan
    s = str(raw_value).strip()
    # Extract the first number in the string
    match = re.search(r'[0-9]+\.?[0-9]*', s)
    if match:
        return float(match.group())
    return np.nan


manifest['pixel_spacing'] = manifest['PixelSpacing'].apply(parse_pixel_spacing)

print('Parsed pixel_spacing values:')
print(manifest['pixel_spacing'].value_counts())
print(f'Missing after parsing: {manifest["pixel_spacing"].isnull().sum()}')

---
## 4. Parse `ImageSize` — standardise formats

Multiple formats exist: `'1670x2010'`, `'[1670, 2010]'`, `'[3015, 2505]'`, etc.  
We extract width and height as separate integer columns.

In [ ]:
def parse_image_size(raw_value):
    """
    Parse ImageSize from its raw string to (width, height) integers.

    Handles:
        '1670x2010'     → (1670, 2010)
        '[1670, 2010]'  → (1670, 2010)
        '[3015, 2505]'  → (3015, 2505)
    Returns (None, None) on failure.
    """
    if pd.isnull(raw_value):
        return None, None
    s = str(raw_value).strip()
    # Find all integers in the string
    nums = re.findall(r'\d+', s)
    if len(nums) >= 2:
        return int(nums[0]), int(nums[1])
    return None, None


manifest[['orig_width', 'orig_height']] = manifest['ImageSize'].apply(
    lambda v: pd.Series(parse_image_size(v))
)

print('Parsed original image dimensions:')
print(manifest[['orig_width', 'orig_height']].drop_duplicates().value_counts().reset_index())
print(f'Missing width : {manifest["orig_width"].isnull().sum()}')
print(f'Missing height: {manifest["orig_height"].isnull().sum()}')

---
## 5. Encode categorical features

We convert text categories to integers. The mapping is saved to `label_encoders.csv` so it can be reproduced exactly at inference time.

**Important rules:**
- Encoders are fit on the **entire dataset** (not just training) because the categories are fixed and known upfront (e.g. F/M/O, L/R, centre names). This is safe — we are encoding labels, not learning data-driven statistics from labels.
- `Normalized PatientID` and `StudyID` are **excluded** — they are identifiers, not features.
- `isRA` is the target — it is kept as-is (0/1 integer).

In [ ]:
# ── Columns to encode ─────────────────────────────────────────────────────────
CATEGORICAL_COLS = ['Sex', 'Center', 'LR', 'laterality']

encoder_records = []    # store the mapping for reproducibility

for col in CATEGORICAL_COLS:
    if col not in manifest.columns:
        print(f'  Column "{col}" not found in manifest — skipping.')
        continue

    le = LabelEncoder()
    # Fill any NaNs with 'Unknown' before encoding so fit() never sees NaN
    manifest[col] = manifest[col].fillna('Unknown').astype(str)
    encoded_col   = col + '_enc'
    manifest[encoded_col] = le.fit_transform(manifest[col])

    # Record the mapping
    for original, numeric in zip(le.classes_, le.transform(le.classes_)):
        encoder_records.append({
            'column'   : col,
            'original' : original,
            'encoded'  : int(numeric),
        })

    print(f'  {col}  →  {encoded_col} : {dict(zip(le.classes_, le.transform(le.classes_)))}')

encoder_df = pd.DataFrame(encoder_records)
encoder_df.to_csv(OUTPUTS_DIR / 'label_encoders.csv', index=False)
print(f'\nEncoder mappings saved → {OUTPUTS_DIR / "label_encoders.csv"}')

---
## 6. Normalise `Age`

Age is a continuous numerical feature. We standardise it using **training-set statistics only** (mean and std).  
This prevents any information from the validation or test sets from influencing the normalisation.

Formula: `age_norm = (age - train_mean) / train_std`

In [ ]:
# ── Compute Age statistics from training set only ─────────────────────────────
train_rows  = manifest[manifest['split'] == 'train']
age_mean    = train_rows['Age'].mean()
age_std     = train_rows['Age'].std()

print(f'Training set Age mean : {age_mean:.4f}')
print(f'Training set Age std  : {age_std:.4f}')

# ── Apply to all splits using training stats ──────────────────────────────────
manifest['age_norm'] = (manifest['Age'] - age_mean) / age_std

print()
print('Age_norm statistics per split:')
print(manifest.groupby('split')['age_norm'].describe().round(4))
print()
print('Note: train mean ≈ 0, std ≈ 1 (as expected). Val and test may differ slightly.')

---
## 7. Image preprocessing — function definition

We define the preprocessing function that will be applied to each image during training.  
This does **not** modify the original files. It operates in memory.

Steps:
1. Open image with Pillow (safe, handles corrupt files)
2. Convert to RGB (ensures 3 channels, handles grayscale BMP)
3. Resize to 224 × 224 (bilinear interpolation)
4. Convert to NumPy array
5. Normalise to [0, 1] by dividing by 255
6. Standardise using training-set mean and std (per channel)

In [ ]:
def preprocess_image(
    image_path: str,
    target_size: tuple = TARGET_SIZE,
    channel_mean: np.ndarray = None,
    channel_std : np.ndarray = None,
) -> np.ndarray:
    """
    Load and preprocess a single radiograph image.

    Parameters
    ----------
    image_path  : str  — absolute path to the .bmp file
    target_size : (W, H) — output dimensions
    channel_mean: np.ndarray shape (3,) — per-channel mean (from training set)
    channel_std : np.ndarray shape (3,) — per-channel std  (from training set)

    Returns
    -------
    np.ndarray of shape (H, W, 3), dtype float32
    Returns None if the image cannot be opened.
    """
    try:
        # Step 1: Open image safely
        with Image.open(image_path) as img:

            # Step 2: Convert to RGB (3 channels)
            # Grayscale images (mode='L') have 1 channel; CNNs expect 3.
            # RGB conversion copies the single channel into all 3 channels.
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # Step 3: Resize to target size
            # BILINEAR gives a good balance of quality and speed.
            img = img.resize(target_size, resample=RESIZE_METHOD)

            # Step 4: Convert to NumPy array (H, W, 3), dtype uint8
            arr = np.array(img, dtype=np.float32)

    except Exception as e:
        print(f'  ERROR loading {image_path}: {e}')
        return None

    # Step 5: Scale to [0, 1]
    arr = arr / PIXEL_SCALE

    # Step 6: Per-channel standardisation (if stats are provided)
    # This centres each channel around 0 with unit variance, which
    # helps gradient-based optimisation converge faster.
    if channel_mean is not None and channel_std is not None:
        arr = (arr - channel_mean) / (channel_std + 1e-8)  # 1e-8 prevents division by zero

    return arr


print('preprocess_image() function defined.')

In [ ]:
# ── Smoke test: preprocess one image and inspect the result ───────────────────
test_row  = manifest.iloc[0]
test_path = test_row['full_path']

print(f'Testing on: {test_row["filename"]}')
print(f'Full path : {test_path}')

# Preprocess without normalisation stats first (just resize + scale)
arr = preprocess_image(test_path)

if arr is not None:
    print(f'\nPreprocessed array shape : {arr.shape}')   # should be (224, 224, 3)
    print(f'dtype                    : {arr.dtype}')
    print(f'Value range              : [{arr.min():.4f}, {arr.max():.4f}]')
    print(f'Mean                     : {arr.mean():.4f}')
    print(f'Std                      : {arr.std():.4f}')
else:
    print('ERROR: Could not preprocess test image.')

In [ ]:
# ── Visualise: original vs preprocessed ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle(f'Image preprocessing demo\n{test_row["filename"]}', fontsize=11)

# Original image (thumbnail for display)
with Image.open(test_path) as orig:
    orig_thumb = orig.copy()
    orig_thumb.thumbnail((400, 400))
    axes[0].imshow(orig_thumb, cmap='gray' if orig.mode == 'L' else None)
    axes[0].set_title(f'Original\n{orig.size[0]}×{orig.size[1]} px | mode={orig.mode}')
    axes[0].axis('off')

# Preprocessed image
axes[1].imshow(np.clip(arr, 0, 1))   # clip for display
axes[1].set_title(f'Preprocessed (224×224, RGB, scaled to [0,1])')
axes[1].axis('off')

plt.tight_layout()
save_path = PLOTS_DIR / '04_preprocessing_demo.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {save_path}')

---
## 8. Compute channel normalisation statistics

We compute the per-channel mean and standard deviation across **all training images**.  
These values are saved and will be used to standardise images during training, validation, and inference.

> **Why training set only?**  
> Using validation or test images to compute normalisation statistics would cause data leakage — the model would be trained with knowledge derived from the test set.

In [ ]:
# ── Compute mean and std from training images ─────────────────────────────────
train_manifest = manifest[manifest['split'] == 'train'].reset_index(drop=True)

print(f'Training images to process: {len(train_manifest)}')
print('This may take 1–3 minutes...')

# We accumulate pixel sums in float64 to avoid overflow
n_pixels  = 0
channel_sum    = np.zeros(3, dtype=np.float64)
channel_sq_sum = np.zeros(3, dtype=np.float64)

failed_images = []

for idx, row in train_manifest.iterrows():
    arr = preprocess_image(row['full_path'])   # returns [0,1] scaled, no standardisation yet

    if arr is None:
        failed_images.append(row['filename'])
        continue

    # arr shape: (H, W, 3)
    h, w, c = arr.shape
    n_pixels      += h * w
    channel_sum    += arr.sum(axis=(0, 1))        # sum over H and W, keep channels
    channel_sq_sum += (arr ** 2).sum(axis=(0, 1)) # sum of squares

    if (idx + 1) % 100 == 0:
        print(f'  Processed {idx + 1} / {len(train_manifest)} training images...')

# ── Final statistics ──────────────────────────────────────────────────────────
channel_mean = channel_sum    / n_pixels
channel_var  = channel_sq_sum / n_pixels - channel_mean ** 2
channel_std  = np.sqrt(np.maximum(channel_var, 0))   # clamp negatives from floating point

print()
print(f'Total pixels processed  : {n_pixels:,}')
print(f'Failed images           : {len(failed_images)}')
print()
print(f'Channel mean (R, G, B)  : {channel_mean}')
print(f'Channel std  (R, G, B)  : {channel_std}')

In [ ]:
# ── Save normalisation statistics ─────────────────────────────────────────────
norm_stats = pd.DataFrame({
    'channel'   : ['R', 'G', 'B'],
    'mean'      : channel_mean,
    'std'       : channel_std,
})

print('Normalisation statistics:')
print(norm_stats.to_string(index=False))

norm_stats_path = OUTPUTS_DIR / 'normalisation_stats.csv'
norm_stats.to_csv(norm_stats_path, index=False)
print(f'\nSaved → {norm_stats_path}')

In [ ]:
# ── Verify: re-run smoke test WITH normalisation stats ────────────────────────
arr_norm = preprocess_image(
    test_path,
    channel_mean=channel_mean,
    channel_std=channel_std
)

if arr_norm is not None:
    print('After standardisation:')
    print(f'  Shape: {arr_norm.shape}')
    print(f'  Range: [{arr_norm.min():.4f}, {arr_norm.max():.4f}]')
    print(f'  Mean : {arr_norm.mean():.4f}  (should be close to 0)')
    print(f'  Std  : {arr_norm.std():.4f}   (should be close to 1)')

---
## 9. Assemble the final feature table

We produce one clean row per image containing:
- Image path (for the PyTorch Dataset to load)
- Target label (`isRA`)
- Split name (`train` / `val` / `test`)
- Patient ID (for stratified splitting, not used as a model feature)
- All safe ML features: `age_norm`, `Sex_enc`, `LR_enc`, `laterality_enc`, `pixel_spacing`, `orig_width`, `orig_height`

Columns explicitly excluded:
- `Normalized PatientID` — identifier, not a feature
- `StudyID` — identifier
- `Mapped Image Stem` — redundant with `full_path`
- `PixelSpacing` — replaced by `pixel_spacing` (parsed float)
- `ImageSize` — replaced by `orig_width`, `orig_height`
- `Age` — replaced by `age_norm`
- Raw categorical columns — replaced by `_enc` versions
- `Center_enc` — kept but flagged as a potential confound

In [ ]:
# ── Select final columns ──────────────────────────────────────────────────────
FINAL_COLS = [
    # ── Identifiers (NOT features — used for loading/splitting) ────────────────
    'filename',
    'full_path',
    'split',
    'Normalized PatientID',   # kept ONLY for patient-stratified splitting

    # ── Target ─────────────────────────────────────────────────────────────────
    'isRA',

    # ── Numerical features ──────────────────────────────────────────────────────
    'age_norm',
    'pixel_spacing',
    'orig_width',
    'orig_height',

    # ── Encoded categorical features ────────────────────────────────────────────
    'Sex_enc',
    'LR_enc',
    'laterality_enc',
    'Center_enc',   # available but treat as confound — use with caution
]

# Only keep columns that actually exist in the manifest
available_cols = [c for c in FINAL_COLS if c in manifest.columns]
missing_cols   = [c for c in FINAL_COLS if c not in manifest.columns]

if missing_cols:
    print(f'WARNING: These expected columns were not found and will be skipped: {missing_cols}')

final_df = manifest[available_cols].copy()

print(f'Final dataset shape: {final_df.shape}')
print(f'Columns kept       : {list(final_df.columns)}')
print()
final_df.head(5)

In [ ]:
# ── Final missing-value check ──────────────────────────────────────────────────
miss = final_df.isnull().sum()
miss_pct = (miss / len(final_df) * 100).round(2)

miss_report = pd.DataFrame({'n_missing': miss, 'missing_pct': miss_pct})
print('Missing values in final dataset:')
print(miss_report[miss_report['n_missing'] > 0].to_string())
if (miss_report['n_missing'] == 0).all():
    print('No missing values — dataset is complete.')

---
## 10. Patient isolation verification

Before saving, we confirm that no patient's images are split across training and test sets.  
The original dataset authors have already attempted to split by patient, but we verify this explicitly.

In [ ]:
# ── Check for train/test patient overlap ──────────────────────────────────────
train_patients = set(final_df[final_df['split'] == 'train']['Normalized PatientID'].unique())
val_patients   = set(final_df[final_df['split'] == 'val'  ]['Normalized PatientID'].unique())
test_patients  = set(final_df[final_df['split'] == 'test' ]['Normalized PatientID'].unique())

train_test_overlap = train_patients & test_patients
train_val_overlap  = train_patients & val_patients
val_test_overlap   = val_patients   & test_patients

print(f'Train patients : {len(train_patients)}')
print(f'Val patients   : {len(val_patients)}')
print(f'Test patients  : {len(test_patients)}')
print()
print(f'Train ∩ Test overlap   : {len(train_test_overlap)} patients')
print(f'Train ∩ Val  overlap   : {len(train_val_overlap)} patients')
print(f'Val   ∩ Test overlap   : {len(val_test_overlap)} patients')

if train_test_overlap:
    print()
    print('LEAKAGE WARNING: These patients appear in BOTH train and test:')
    print(sorted(train_test_overlap))
    print()
    print('RECOMMENDATION: The dataset uses the original splits provided by the authors.')
    print('These shared patients appear in the official split as-is.')
    print('For rigorous evaluation, consider re-splitting with strict patient stratification.')
else:
    print()
    print('Train and Test sets have NO patient overlap — safe to use for evaluation.')

In [ ]:
# ── Final class distribution check ────────────────────────────────────────────
label_map = {0: 'Non-RA', 1: 'RA'}

print('Final class distribution per split:')
for split_name in ['train', 'val', 'test']:
    subset = final_df[final_df['split'] == split_name]
    counts = subset['isRA'].map(label_map).value_counts()
    total  = len(subset)
    ra_pct = counts.get('RA', 0) / total * 100
    print(f'  {split_name:<6}: {counts.get("Non-RA", 0):>3} Non-RA  |  {counts.get("RA", 0):>4} RA  '
          f'|  Total: {total}  |  RA%: {ra_pct:.1f}%')

---
## 11. Save per-split manifest files

In [ ]:
# ── Save train / val / test manifest CSVs ────────────────────────────────────
for split_name in ['train', 'val', 'test']:
    subset    = final_df[final_df['split'] == split_name].copy()
    out_path  = OUTPUTS_DIR / f'{split_name}_manifest.csv'
    subset.to_csv(out_path, index=False)
    print(f'{split_name} manifest saved → {out_path}  ({len(subset)} rows)')

print()
print('These CSVs are the primary input for the PyTorch Dataset class in future notebooks.')

---
## 12. How the PyTorch Dataset will use these outputs

For reference, here is the interface the preprocessing outputs are designed for.

In [ ]:
# ── Skeleton of the future PyTorch Dataset (NOT executed yet) ─────────────────
# This is here for documentation and planning purposes only.
# The actual training pipeline will be built in a later notebook.

skeleton_code = '''
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from PIL import Image


class RADataset(Dataset):
    """
    PyTorch Dataset for the RAM-H1200-v1 RA dataset.
    Loads images on-the-fly and applies preprocessing.
    """

    def __init__(self, manifest_csv, norm_stats_csv, target_size=(224, 224), transform=None):
        self.df          = pd.read_csv(manifest_csv)
        norm             = pd.read_csv(norm_stats_csv)
        self.ch_mean     = norm['mean'].values.astype(np.float32)
        self.ch_std      = norm['std'].values.astype(np.float32)
        self.target_size = target_size
        self.transform   = transform   # optional torchvision augmentations

        # Metadata features to include alongside the image
        self.meta_cols = ['age_norm', 'Sex_enc', 'laterality_enc', 'pixel_spacing']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Load and preprocess image ─────────────────────────────────────────
        with Image.open(row['full_path']) as img:
            img = img.convert('RGB')
            img = img.resize(self.target_size, Image.BILINEAR)
            arr = np.array(img, dtype=np.float32) / 255.0
            arr = (arr - self.ch_mean) / (self.ch_std + 1e-8)

        # HWC → CHW (PyTorch convention)
        image_tensor = torch.from_numpy(arr.transpose(2, 0, 1))

        if self.transform:
            image_tensor = self.transform(image_tensor)

        # ── Metadata features ─────────────────────────────────────────────────
        meta_values = row[self.meta_cols].values.astype(np.float32)
        meta_tensor = torch.from_numpy(meta_values)

        # ── Label ─────────────────────────────────────────────────────────────
        label = torch.tensor(int(row['isRA']), dtype=torch.long)

        return image_tensor, meta_tensor, label
'''

print('PyTorch Dataset skeleton (for future reference):')
print(skeleton_code)

---
## 13. Final preprocessing summary

In [ ]:
# ── Build the preprocessing report ───────────────────────────────────────────
report = {
    # Image preprocessing
    'target_image_size'              : f'{TARGET_SIZE[0]}x{TARGET_SIZE[1]}',
    'resize_method'                  : 'BILINEAR (PIL)',
    'pixel_scale'                    : '÷ 255  → [0, 1]',
    'channel_normalisation'          : 'mean/std from training set',
    'channel_mean_R'                 : round(float(channel_mean[0]), 6),
    'channel_mean_G'                 : round(float(channel_mean[1]), 6),
    'channel_mean_B'                 : round(float(channel_mean[2]), 6),
    'channel_std_R'                  : round(float(channel_std[0]), 6),
    'channel_std_G'                  : round(float(channel_std[1]), 6),
    'channel_std_B'                  : round(float(channel_std[2]), 6),
    'corrupted_images_removed'       : len(corrupted_files),

    # Metadata preprocessing
    'age_normalisation'              : 'z-score (train mean/std)',
    'age_train_mean'                 : round(age_mean, 4),
    'age_train_std'                  : round(age_std, 4),
    'pixel_spacing_parsing'          : 'string → float (first number extracted)',
    'image_size_parsing'             : 'string → (orig_width, orig_height) integers',
    'categorical_encoding'           : 'LabelEncoder (Sex, Center, LR, laterality)',
    'patient_id_in_features'         : 'NO — used only for split verification',
    'study_id_in_features'           : 'NO — excluded',

    # Final split sizes
    'final_train_images'             : int((final_df['split'] == 'train').sum()),
    'final_val_images'               : int((final_df['split'] == 'val').sum()),
    'final_test_images'              : int((final_df['split'] == 'test').sum()),
    'train_test_patient_overlap'     : len(train_test_overlap),
}

report_df = pd.DataFrame.from_dict(report, orient='index', columns=['value'])
print('=== Preprocessing Report ===')
print(report_df.to_string())

report_df.to_csv(OUTPUTS_DIR / '04_preprocessing_report.csv')
print(f'\nReport saved → {OUTPUTS_DIR / "04_preprocessing_report.csv"}')

In [ ]:
# ── Final confirmation of all saved outputs ───────────────────────────────────
print('=== All outputs from Notebook 04 ===')
saved_files = [
    OUTPUTS_DIR / 'train_manifest.csv',
    OUTPUTS_DIR / 'val_manifest.csv',
    OUTPUTS_DIR / 'test_manifest.csv',
    OUTPUTS_DIR / 'normalisation_stats.csv',
    OUTPUTS_DIR / 'label_encoders.csv',
    OUTPUTS_DIR / '04_preprocessing_report.csv',
    PLOTS_DIR   / '04_preprocessing_demo.png',
]
for f in saved_files:
    status = 'OK' if f.exists() else 'MISSING'
    print(f'  [{status}]  {f.name}')

---
## Summary

The preprocessing pipeline is complete. Here is what was done and why:

| Step | Action | Reason |
|---|---|---|
| Image loading | Pillow `Image.open()` with error handling | Safe; prevents crashes on corrupt files |
| Colour conversion | `img.convert('RGB')` | Ensures 3-channel input for pretrained CNNs |
| Resize | 224 × 224 bilinear | Standard CNN input size; works with ImageNet pretrained weights |
| Scale | ÷ 255 → [0, 1] | Brings pixel values into a standard range |
| Standardise | (x − μ) / σ using training stats | Zero-centres data; accelerates learning |
| Age normalisation | z-score from training set | Prevents scale mismatch in multimodal fusion |
| Categorical encoding | `LabelEncoder` | Converts text to integers for neural net input |
| `PixelSpacing` parsing | String → float | Makes the acquisition parameter usable |
| Patient ID isolation | Excluded from features | Prevents data leakage |

**Files ready for training:**
- `outputs/reports/train_manifest.csv`
- `outputs/reports/val_manifest.csv`
- `outputs/reports/test_manifest.csv`
- `outputs/reports/normalisation_stats.csv`
- `outputs/reports/label_encoders.csv`

The original dataset files in `Dataset/ra/` have **not been modified**.  
All preprocessing is applied in-memory at load time.